# Agents are your collegues. You do not want a forgetful agent !

![title](./images/momento.jpeg)



A step-by-step tutorial on **agent memory** using the **Anthropic Python SDK** (Claude) and **OpenAI** (embeddings).

We progress through 5 stages — each one solves a concrete failure of the previous:

| Stage | Memory Type | What it solves |
|-------|-------------|----------------|
| 1 | ❌ No memory | Baseline — dumb agent |
| 2 | 🧠 In-context history | Remembers only *within* a session, but not efficient |
| 3 | 📝 Summarization | Remembers only within session efficiently|
| 4 | 💾 File persistence + Tool Use | Remembers between sessions, but not very efficiently |
| 5 | 🔍 Semantic (vector store) | Remembers between sessions, memory efficiently |

**Scenario throughout:** a research assistant you tell facts to. By Stage 5 it remembers across sessions and retrieves only the most relevant context automatically.


In [1]:
!pip install -q anthropic openai numpy

zsh:1: command not found: pip


In [2]:
import json
import numpy as np
from pathlib import Path
from datetime import datetime
from getpass import getpass
import anthropic
from openai import OpenAI


In [3]:
anthropic_api_key = getpass("Enter your Anthropic API key: ")
openai_api_key    = getpass("Enter your OpenAI API key: ")


Enter your Anthropic API key:  ········
Enter your OpenAI API key:  ········


In [4]:
anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)
openai_client    = OpenAI(api_key=openai_api_key)

MODEL = "claude-haiku-4-5-20251001"   # fast & cost-effective for demos
print(f"Ready. Model: {MODEL}")


Ready. Model: claude-haiku-4-5-20251001


---
## Stage 1 — The Amnesiac Agent (No Memory)

Every API call is completely stateless. The agent sees only the current message and forgets everything the moment the call returns.


![title](./images/no_memory.jpg)

In [5]:
def chat_no_memory(user_message: str) -> str:
    response = anthropic_client.messages.create(
        model=MODEL,
        max_tokens=512,
        system="You are a helpful personal assistant.",
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text

print("=== Stage 1: No Memory ===\n")
r1 = chat_no_memory("My name is Alex and I am writing a paper on attention mechanisms in transformers.")
print(f"Turn 1:\n{r1}\n")

r2 = chat_no_memory("What is my name?")
print(f"Turn 2 — 'What is my name?':\n{r2}\n")

r3 = chat_no_memory("What topic am I researching?")
print(f"Turn 3 — 'What topic am I researching?':\n{r3}")


=== Stage 1: No Memory ===

Turn 1:
# Great topic! I'd be happy to help you with your paper on attention mechanisms in transformers.

Here are some key areas I can assist with:

## Core Concepts
- **Self-attention mechanics** – how queries, keys, and values work together
- **Scaled dot-product attention** – the mathematical foundation
- **Multi-head attention** – why multiple representation subspaces matter

## Advanced Topics
- Different attention variants (cross-attention, causal attention, sparse attention)
- Computational complexity and efficiency improvements
- Positional encoding and its role in attention

## Practical Aspects
- How attention enables parallel processing
- Comparison to RNNs and CNNs
- Real-world applications and performance

## Paper Structure Help
- Literature review organization
- Explaining mathematical notation clearly
- Creating effective visualizations or diagrams

**What would be most helpful for you right now?** For example:
- Do you need help with a spec

**Problem:** The agent has no memory of Turn 1 when Turn 2 is asked. Each call starts fresh.

Stage 2 fixes this by passing the full conversation history on every call.


---
## Stage 2 — Working Memory (In-Context History)

We maintain a `history` list and pass it on every call. The agent now remembers within a session.


![title](./images/in_context_memory.jpg)

In [6]:
class InContextMemoryAgent:
    """Maintains full conversation history within a session."""

    def __init__(self, system_prompt: str = "You are a helpful personal assistant."):
        self.system_prompt = system_prompt
        self.history = []

    def chat(self, user_message: str) -> str:
        self.history.append({"role": "user", "content": user_message})
        response = anthropic_client.messages.create(
            model=MODEL,
            max_tokens=512,
            system=self.system_prompt,
            messages=self.history,
        )
        reply = response.content[0].text
        self.history.append({"role": "assistant", "content": reply})
        return reply

    def reset(self):
        self.history = []

    @property
    def approx_words(self) -> int:
        return sum(len(m["content"].split()) for m in self.history)


In [7]:
agent2 = InContextMemoryAgent()

print("=== Stage 2: In-Context Memory ===\n")

r1 = agent2.chat("My name is Alex. I am a PhD student at MIT writing a paper on attention mechanisms. Deadline: July 15.")
print(f"Turn 1:\n{r1}\n")

r2 = agent2.chat("What is my name and when is my deadline?")
print(f"Turn 2:\n{r2}\n")

r3 = agent2.chat("Suggest one related subtopic I could add to my paper.")
print(f"Turn 3:\n{r3}\n")

print(f"History: {len(agent2.history)} messages | ~{agent2.approx_words} words in context")


=== Stage 2: In-Context Memory ===

Turn 1:
# Hello Alex! 👋

Great to meet you. I've noted your context:
- **Your role:** PhD student at MIT
- **Research focus:** Attention mechanisms
- **Paper deadline:** July 15

I'm here to help you with your research and writing. I can assist with:

- **Conceptual clarity** – explaining or refining attention mechanism concepts
- **Literature review** – discussing relevant papers and developments in the field
- **Writing** – drafting sections, improving clarity, structuring arguments
- **Technical details** – working through mathematical formulations or implementations
- **Problem-solving** – brainstorming approaches to research challenges

**Quick questions to get started:**
1. What specific aspect of attention mechanisms is your paper focusing on?
2. What stage are you at (early draft, revision, etc.)?
3. What's your biggest priority right now?

Feel free to share what you're working on whenever you're ready!

Turn 2:
Your name is **Alex** and you

**Problem:** Context windows are finite. Every token in history costs money and eventually hits the limit.

- Claude Haiku supports ~200K tokens — big, but long agentic sessions or document-heavy chats fill it fast.
- Older, irrelevant turns dilute attention on what matters now.

Stage 3 compresses old context with **summarization**.


---
## Stage 3 — Compressed Memory (Summarization)

When history exceeds `max_turns`, we compress the oldest half into a bullet-point summary.
Recent turns stay verbatim; older context is distilled and prepended to the system prompt.


![title](./images/summarization.jpg)

In [8]:
class SummarizationMemoryAgent:
    """Compresses old context into a running summary when history grows too long."""

    def __init__(
        self,
        system_prompt: str = "You are a helpful personal assistant.",
        max_turns: int = 6,
    ):
        self.system_prompt = system_prompt
        self.history = []
        self.summary = ""
        self.max_turns = max_turns
        self._compress_count = 0

    def _compress(self):
        half = self.max_turns // 2
        old_turns, self.history = self.history[:half], self.history[half:]

        conversation_text = "\n".join(
            f"{m['role'].upper()}: {m['content']}" for m in old_turns
        )
        resp = anthropic_client.messages.create(
            model=MODEL,
            max_tokens=256,
            messages=[{
                "role": "user",
                "content": (
                    "Summarize the key facts from this conversation in 3-5 bullet points:\n\n"
                    + conversation_text
                ),
            }],
        )
        new_bullets = resp.content[0].text
        self.summary = (
            f"{self.summary}\n{new_bullets}".strip() if self.summary else new_bullets
        )
        self._compress_count += 1
        print(f"  [Compressed {half} turns into summary (pass #{self._compress_count})]")

    def chat(self, user_message: str) -> str:
        if len(self.history) >= self.max_turns:
            self._compress()

        self.history.append({"role": "user", "content": user_message})

        system = self.system_prompt
        if self.summary:
            system += f"\n\nMemory summary from earlier in this conversation:\n{self.summary}"

        resp = anthropic_client.messages.create(
            model=MODEL,
            max_tokens=512,
            system=system,
            messages=self.history,
        )
        reply = resp.content[0].text
        self.history.append({"role": "assistant", "content": reply})
        return reply

    def show_state(self):
        print(f"  Active history : {len(self.history)} messages")
        if self.summary:
            print(f"  Summary:\n{self.summary}")
        else:
            print("  Summary        : (none yet)")


In [9]:
agent3 = SummarizationMemoryAgent(max_turns=4)  # low threshold to trigger compression quickly

turns = [
    "My name is Alex. I am a PhD student at MIT.",
    "I am researching attention mechanisms in transformers.",
    "My advisor is Prof. Chen. My paper deadline is July 15.",
    "I enjoy hiking and photography in my free time.",
    "What do you know about my research so far?",   # triggers compression
    "Can you remind me of my paper deadline?",
]

print("=== Stage 3: Summarization Memory ===\n")
for i, msg in enumerate(turns, 1):
    print(f"Turn {i}: {msg}")
    reply = agent3.chat(msg)
    print(f"  Agent: {reply[:180]}...\n")

print("--- Final state ---")
agent3.show_state()


=== Stage 3: Summarization Memory ===

Turn 1: My name is Alex. I am a PhD student at MIT.
  Agent: Nice to meet you, Alex! It's great to learn about you. Being a PhD student at MIT is a significant achievement—that's an excellent institution with a strong reputation across many ...

Turn 2: I am researching attention mechanisms in transformers.
  Agent: That's a fascinating research area! Attention mechanisms are central to modern deep learning, and there's still a lot of interesting work happening around them. Here are some areas...

Turn 3: My advisor is Prof. Chen. My paper deadline is July 15.
  [Compressed 2 turns into summary (pass #1)]
  Agent: Good to know! That gives you a concrete timeline to work toward. A few follow-up questions to help me support you effectively:

**On your research:**
- What's your main focus withi...

Turn 4: I enjoy hiking and photography in my free time.
  [Compressed 2 turns into summary (pass #2)]
  Agent: That's a nice balance! Hiking and photograp

**What happened:** When Turn 5 was asked, the agent had 4 messages (max). Before answering, it compressed Turns 1–2 into bullet points and moved them to the system prompt. Recent turns stayed verbatim.

**Problem still remaining:** The summary and history exist *only in RAM*. When this Python process ends, all memory is gone.

Stage 4 solves cross-session persistence with file storage and Claude **tool use**.


---
## Stage 4 — Episodic Memory (File Persistence + Tool Use)

We give the agent two tools:

| Tool | What it does |
|------|-------------|
| `save_memory` | Writes a fact to a JSON file on disk |
| `recall_memories` | Reads all saved facts back |

The agent **decides autonomously** what is worth saving — this is Claude's native tool-use (function calling) pattern, central to the Anthropic Agent SDK approach.


![title](./images/file_persistance.jpg)

In [10]:
MEMORY_FILE = Path("agent_memory.json")

def _load_store() -> dict:
    if MEMORY_FILE.exists():
        return json.loads(MEMORY_FILE.read_text())
    return {"facts": []}

def _persist_store(store: dict):
    MEMORY_FILE.write_text(json.dumps(store, indent=2))

MEMORY_TOOLS = [
    {
        "name": "save_memory",
        "description": "Save an important fact about the user to persistent memory across sessions.",
        "input_schema": {
            "type": "object",
            "properties": {
                "fact": {
                    "type": "string",
                    "description": "The fact to remember, as a clear sentence.",
                },
                "category": {
                    "type": "string",
                    "enum": ["personal", "project", "preference"],
                },
            },
            "required": ["fact", "category"],
        },
    },
    {
        "name": "recall_memories",
        "description": "Retrieve all saved memories about the user.",
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
]


class PersistentMemoryAgent:
    """Agent with file-backed persistent memory via Claude tool use."""

    def __init__(self):
        self.store = _load_store()
        self.history = []

    def _system_prompt(self) -> str:
        facts = "\n".join(
            f"  - [{m['category']}] {m['fact']}" for m in self.store["facts"]
        )
        return (
            "You are a helpful personal assistant with persistent memory.\n"
            "When the user shares something important (name, project, deadline, preference), "
            "call save_memory immediately so it survives future sessions.\n"
            "Use recall_memories if you are unsure what you already know.\n\n"
            "Known memories:\n" + (facts if facts else "  (none yet)")
        )

    def _handle_tool(self, name: str, inp: dict) -> str:
        if name == "save_memory":
            self.store["facts"].append(
                {"fact": inp["fact"], "category": inp["category"], "saved_at": datetime.now().isoformat()}
            )
            _persist_store(self.store)
            return f"Saved: {inp['fact']}"
        if name == "recall_memories":
            if not self.store["facts"]:
                return "No memories saved yet."
            return "\n".join(f"[{m['category']}] {m['fact']}" for m in self.store["facts"])
        return "Unknown tool."

    def chat(self, user_message: str) -> str:
        self.history.append({"role": "user", "content": user_message})

        while True:
            resp = anthropic_client.messages.create(
                model=MODEL,
                max_tokens=1024,
                system=self._system_prompt(),
                tools=MEMORY_TOOLS,
                messages=self.history,
            )

            if resp.stop_reason == "tool_use":
                tool_results = []
                for block in resp.content:
                    if block.type == "tool_use":
                        result = self._handle_tool(block.name, block.input)
                        print(f"  [tool] {block.name}({block.input}) → {result}")
                        tool_results.append(
                            {"type": "tool_result", "tool_use_id": block.id, "content": result}
                        )
                self.history.append({"role": "assistant", "content": resp.content})
                self.history.append({"role": "user", "content": tool_results})
            else:
                reply = next(b.text for b in resp.content if hasattr(b, "text"))
                self.history.append({"role": "assistant", "content": reply})
                return reply

    def show_store(self):
        for m in self.store["facts"]:
            print(f"  [{m['category']}] {m['fact']}")


In [11]:
# Start fresh for the demo
if MEMORY_FILE.exists():
    MEMORY_FILE.unlink()

print("=== Stage 4: Persistent Memory + Tool Use ===\n")

print("--- Session 1 ---")
agent4a = PersistentMemoryAgent()
r1 = agent4a.chat("Hi! I am Alex, a PhD student at MIT. I am working on a paper about transformer attention mechanisms. Deadline is July 15 and my advisor is Prof. Chen.")
print(f"Agent: {r1}\n")

r2 = agent4a.chat("I prefer concise, bullet-point style responses.")
print(f"Agent: {r2}\n")

print("\n--- Session 2 (new agent instance, same memory file) ---")
agent4b = PersistentMemoryAgent()   # loads from disk automatically
r3 = agent4b.chat("Do you remember anything about me?")
print(f"Agent: {r3}\n")

print("\n--- Memory on disk ---")
agent4b.show_store()


=== Stage 4: Persistent Memory + Tool Use ===

--- Session 1 ---
  [tool] save_memory({'fact': 'Alex is a PhD student at MIT.', 'category': 'personal'}) → Saved: Alex is a PhD student at MIT.
  [tool] save_memory({'fact': "Alex's advisor is Prof. Chen.", 'category': 'personal'}) → Saved: Alex's advisor is Prof. Chen.
  [tool] save_memory({'fact': 'Alex is working on a paper about transformer attention mechanisms.', 'category': 'project'}) → Saved: Alex is working on a paper about transformer attention mechanisms.
  [tool] save_memory({'fact': "The deadline for Alex's transformer attention mechanisms paper is July 15.", 'category': 'project'}) → Saved: The deadline for Alex's transformer attention mechanisms paper is July 15.
Agent: Perfect! I've saved all the details about you and your project. I now know that you're a PhD student at MIT working on a transformer attention mechanisms paper with Prof. Chen as your advisor, and the paper is due on July 15.

How can I help you with your re

**What happened:**
- Session 1: Claude called `save_memory` autonomously after you introduced yourself.
- Session 2: A *brand-new* agent instance loaded the JSON file and answered correctly.

**Remaining problem:** Loading *all* stored facts into the system prompt works now, but won't scale to hundreds of memories. Stage 5 retrieves only the most *relevant* ones.


---
## Stage 5 — Semantic Memory (Vector Store)

Each memory is **embedded** (using OpenAI `text-embedding-3-small`) and stored as a vector.
At query time, we embed the question and retrieve the **top-K most similar memories** via cosine similarity — a minimal vector store with no extra dependencies.


![title](./images/semantic.jpg)

In [12]:
def embed(text: str) -> np.ndarray:
    resp = openai_client.embeddings.create(model="text-embedding-3-small", input=text)
    return np.array(resp.data[0].embedding, dtype=np.float32)

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


class SemanticMemoryAgent:
    """Agent backed by a numpy vector store for semantic memory retrieval."""

    def __init__(self, top_k: int = 3):
        self.top_k = top_k
        self.store: list[dict] = []   # {"text": str, "vec": np.ndarray}
        self.history = []

    def remember(self, text: str):
        """Embed and store a memory (called manually here; could be a tool in production)."""
        self.store.append({"text": text, "vec": embed(text)})
        print(f"  [stored] {text}")

    def retrieve(self, query: str) -> list[str]:
        if not self.store:
            return []
        q_vec = embed(query)
        scored = [(cosine_sim(q_vec, m["vec"]), m["text"]) for m in self.store]
        scored.sort(reverse=True)
        return [text for _, text in scored[: self.top_k]]

    def chat(self, user_message: str) -> str:
        relevant = self.retrieve(user_message)

        system = "You are a helpful personal assistant with semantic memory."
        if relevant:
            system += "\n\nMost relevant memories retrieved for this query:\n" + "\n".join(
                f"- {m}" for m in relevant
            )

        self.history.append({"role": "user", "content": user_message})
        resp = anthropic_client.messages.create(
            model=MODEL,
            max_tokens=512,
            system=system,
            messages=self.history,
        )
        reply = resp.content[0].text
        self.history.append({"role": "assistant", "content": reply})
        return reply


In [13]:
agent5 = SemanticMemoryAgent(top_k=2)

print("=== Stage 5: Semantic Memory ===\n")
print("--- Loading 10 memories (simulating a full past history) ---")
memory_bank = [
    "Alex is a PhD student at MIT studying NLP.",
    "Alex is writing a paper on attention mechanisms in transformers.",
    "Alex's paper deadline is July 15, 2025.",
    "Alex's faculty advisor is Prof. Chen.",
    "Alex enjoys hiking and photography.",
    "Alex prefers concise, bullet-point style responses.",
    "Alex has a cat named Pixel.",
    "Alex's favourite programming language is Python.",
    "Alex attended NeurIPS 2024 in Vancouver.",
    "Alex is learning to play guitar.",
]
for m in memory_bank:
    agent5.remember(m)

print()
queries = [
    "What is my paper deadline?",
    "What are my hobbies outside of research?",
    "What conferences have I been to?",
]
for q in queries:
    retrieved = agent5.retrieve(q)
    print(f"Query  : {q}")
    print(f"Retrieved: {retrieved}")
    reply = agent5.chat(q)
    print(f"Agent  : {reply}\n")


=== Stage 5: Semantic Memory ===

--- Loading 10 memories (simulating a full past history) ---
  [stored] Alex is a PhD student at MIT studying NLP.
  [stored] Alex is writing a paper on attention mechanisms in transformers.
  [stored] Alex's paper deadline is July 15, 2025.
  [stored] Alex's faculty advisor is Prof. Chen.
  [stored] Alex enjoys hiking and photography.
  [stored] Alex prefers concise, bullet-point style responses.
  [stored] Alex has a cat named Pixel.
  [stored] Alex's favourite programming language is Python.
  [stored] Alex attended NeurIPS 2024 in Vancouver.
  [stored] Alex is learning to play guitar.

Query  : What is my paper deadline?
Retrieved: ["Alex's paper deadline is July 15, 2025.", 'Alex is writing a paper on attention mechanisms in transformers.']
Agent  : Your paper deadline is **July 15, 2025**.

Is there anything you'd like help with regarding your paper on attention mechanisms in transformers?

Query  : What are my hobbies outside of research?
Retrie

---
## Summary — Choosing the Right Memory Strategy

| Memory Type | Persists across sessions | Scales to large memory | Best for |
|-------------|:------------------------:|:---------------------:|----------|
| **No memory** | ❌ | — | Single-turn Q&A |
| **In-context** | ❌ | ❌ | Short conversations |
| **Summarization** | ❌ | ✅ (within session) | Long single-session chats |
| **File + Tool Use** | ✅ | ❌ (loads all facts) | Personal assistants, moderate memory |
| **Semantic (vector)** | ✅ | ✅ | Large memory banks, RAG-style agents |

### Key Takeaways

1. **In-context** is zero-cost and zero-setup — start here.
2. **Summarization** extends effective context by distilling old turns; easy to layer on top.
3. **Tool use (save/recall)** is the Anthropic-native agentic pattern — Claude decides what to save without explicit instruction.
4. **Semantic retrieval** is essential when memories grow large — inject only what's relevant.
5. **Production architecture** combines all four: semantic retrieval for long-term facts, recent verbatim turns, summarization for mid-term.

### Next Steps
- Persist vector embeddings to disk with `np.save` / `np.load`
- Replace the numpy store with ChromaDB, Pinecone, or Weaviate for production scale
- Add a **forgetting** mechanism — time-decay scoring or explicit eviction
- Build a **memory consolidation** step that merges or deduplicates related facts
- Let the agent **self-manage** its vector store using tool use (Stage 4 + Stage 5 combined)
